# Training tutorial: metric matching on a 2-D toy manifold

This notebook trains a metric-matching model from scratch on a "squiggly
circle": a sine wave wrapped around a circle,
$r(\theta) = R + 0.3\sin(f\theta)$. It takes a few minutes. The curve is a
1-dimensional manifold in $\mathbb{R}^2$, so we know in closed form
everything the model is supposed to learn, and we can check its answers:

1. the learned metric $G(x, h) = U^\top U$ should be a rank-1 projector onto
   the tangent line of the curve (so $\lambda_2 \ll \lambda_1$),
2. its top eigenvector should agree with the analytic tangent,
3. as the bandwidth $h$ grows, the fine wiggles should get smoothed away and
   the metric should describe the coarse-scale geometry instead.

Before training anything we compute the classical k-NN kernel estimate of
the CDC. That is the graph-based baseline that metric matching amortizes,
and it gives us a picture to compare the network against.

The training loop is the same thing `scripts/train.py` does, just inlined:
an `MMSystem` driven by `ConditionalMetricMatching`, fit with a plain
Lightning `Trainer`. You need `pip install -e .` from the repo root first.

In [ ]:
import matplotlib.pyplot as plt
import torch
from lightning.pytorch import Trainer, seed_everything

from metric_matching.data.synth.generator import generate_circle_sine_wave_aligned

import warnings
warnings.filterwarnings("ignore", message=".*srun.*")

FREQUENCY = 5      # number of wiggles around the circle
NUM_POINTS = 2000
RADIUS = 1.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
seed_everything(0, workers=True)

pts = generate_circle_sine_wave_aligned(
    num_points=NUM_POINTS, frequency=FREQUENCY, radius=RADIUS, sampler="random"
)

plt.figure(figsize=(4.4, 4.4))
plt.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5, color="0.35")
plt.gca().set_aspect("equal")
plt.title(f"Squiggly circle (frequency = {FREQUENCY})")
plt.show()

## Warm-up: the classical k-NN estimate of the CDC

The CDC matrix can be estimated directly from samples (Sec. 3 of the paper).
At a query point, take its $k$ nearest neighbours, weight them by a kernel of
their distance at bandwidth $h$, and average the outer products of the
differences:
$\hat\Gamma(y) \approx \sum_j w_j (y - x_j)(y - x_j)^\top / h^2$.
With `TorchBruteKNN.query_cdc` this is three lines. We draw the estimate as
ellipses, with axes along the eigenvectors and lengths $\sqrt{\lambda_i}$.
They come out thin and aligned with the curve, which is the geometry the
network is about to learn.

This estimator is the baseline in the paper's scalability comparison
(Fig. 2). Its tangent directions are essentially exact here, but it needs the
whole dataset and a neighbour search at every query, and that is what breaks
down in high dimensions. Metric matching replaces all of it with one forward
pass.

Note that in this low dimensional and high data setting, the graph/kernel-based estimator are near perfect. This is precisely where these methods shine. However, when the dimension grows they start to struggle due to the curse of dimensionality, this is the motivation between the metric matching neural networks!

In [ ]:
from matplotlib.patches import Ellipse

from metric_matching.classical.knn_cdc import TorchBruteKNN


def eigs_desc(G):
    """Eigen-decompose symmetric 2x2 matrices, eigenvalues descending."""
    evals, evecs = torch.linalg.eigh(G)  # ascending
    return evals.flip(-1), evecs.flip(-1)


def draw_ellipses(ax, points, evals, evecs, scale=0.18, **kwargs):
    for p, lam, V in zip(points, evals, evecs):
        angle = torch.rad2deg(torch.atan2(V[1, 0], V[0, 0]))
        ax.add_patch(
            Ellipse(
                p.tolist(),
                width=2 * scale * lam[0].sqrt(),
                height=2 * scale * lam[1].sqrt().clamp(min=1e-3),
                angle=float(angle),
                fill=False,
                **kwargs,
            )
        )


# Query points spread evenly along the curve, for display.
show = generate_circle_sine_wave_aligned(num_points=40, frequency=FREQUENCY, radius=RADIUS)

cdc_knn, _, _ = TorchBruteKNN().fit(pts).query_cdc(show, k=16, h=0.1)
evals_knn, evecs_knn = eigs_desc(cdc_knn)
ratio = (evals_knn[:, 1] / evals_knn[:, 0]).median()
print(f"k-NN estimate: median lambda2/lambda1 = {ratio:.1e} "
      "(the ellipses are so thin they trace the curve itself)")

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(pts[:, 0], pts[:, 1], s=0.5, alpha=0.05, color="0.6")
draw_ellipses(ax, show, evals_knn, evecs_knn, color="teal", lw=1.2)
ax.set_aspect("equal")
ax.set_title("k-NN CDC estimate at h = 0.1 (k = 16) - ellipses are near-degenerate")
plt.show()

## Configure the model

Same config schema as `configs/sphere.yaml`, shrunk to 2-D: a small
FiLM-conditioned residual MLP that predicts a rank-2 factor
$U(x, h) \in \mathbb{R}^{2\times 2}$. The manifold is 1-D, so we expect the
model to end up using only one of the two directions. Bandwidths are sampled
log-normally between `h_min` and `h_max`; the scales that matter here are the
wavelength of the wiggles ($\approx 2\pi R/f \approx 1.3$) and their
amplitude (0.3).


In [ ]:
from metric_matching.systems.mm_system import MMSystem

cfg = {
    "model": {
        "name": "mlp",
        "params": {
            "input_dim": 2,
            "hidden_dim": 512,
            "num_layers": 3,
            "rank": 2,
            "output_dim": 2,
            "time_embedding": True,
        },
    },
    "loss": {
        "h_min": 0.005,
        "h_max": 1.0,
        "sampling_method": "lognormal",
        "smart_training": True,
        "mean_centered": False,
    },
    "optimizer": {
        "name": "adamw",
        "lr": 1e-3,
        "weight_decay": 0.0,
        "lr_scheduler": "cosine",
    },
}

model = MMSystem(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params / 1e3:.0f}k parameters")

## Train

A plain Lightning fit on `(x, dummy_label)` batches. Lightning's built-in
progress bar shows batch progress and training loss within each epoch.
A small callback records epoch losses for the final training curve.
About five minutes on a GPU.

Note that the loss does not go to zero. The conditional loss equals the
intractable marginal loss plus an irreducible variance floor (Theorem 3.1 of
the paper), so it plateaus at a high, noisy value while the metric underneath
keeps improving. We judge the model by the geometry checks below, not by the
loss curve.

In [ ]:
from lightning.pytorch.callbacks import Callback
from torch.utils.data import DataLoader, TensorDataset


class LossHistory(Callback):
    def __init__(self):
        self.losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        self.losses.append(float(trainer.callback_metrics["train/cond_mse_epoch"]))

history = LossHistory()
loader = DataLoader(
    TensorDataset(pts, torch.zeros(len(pts))), batch_size=256, shuffle=True
)
trainer = Trainer(
    max_epochs=3000,
    accelerator="gpu" if str(DEVICE).startswith("cuda") else "cpu",
    devices=1 if str(DEVICE).startswith("cuda") else 1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
    callbacks=[history],
)
trainer.fit(model, train_dataloaders=loader)

plt.figure(figsize=(4.6, 3))
plt.semilogy(range(1, len(history.losses) + 1), history.losses, color="crimson", lw=1)
plt.xlabel("epoch")
plt.ylabel("conditional matching loss")
plt.title("Training curve")
plt.show()


## The learned metric, drawn as ellipses (paper Fig. 1, top left)

At each point we evaluate $G(x, h) = U^\top U$, a $2\times 2$ matrix, and
draw its unit ball: an ellipse with axes along the eigenvectors and lengths
$\sqrt{\lambda_i}$. If the model has learned the geometry, the ellipses
should be thin and lie along the curve, like the k-NN estimate above. The
difference is that each one comes from a single forward pass, with no
dataset and no neighbour search involved.

We observe that the learned geometry is meaningful. Namely, the semi-major axis (longer radius) does correspond to the tangent direction, while the semi-minor axis corresponds to the normal direction.  However, note that the results are not quite as crisp as the kernel method above. This is because this is a high data low dimensional example, which is the setting in which graph/kernel methods shine.

In [ ]:
model = model.to(DEVICE).eval()


def metric_eigs(points, h_value):
    """Eigen-decompose G = U^T U at the given points and bandwidth."""
    x = points.to(DEVICE)
    h = torch.full((len(x),), h_value, device=DEVICE)
    with torch.no_grad():
        U = model(h, x)        # [N, 2, 2]
    G = U.transpose(1, 2) @ U  # tiny, so materializing is fine
    return eigs_desc(G.cpu())


evals, evecs = metric_eigs(show, h_value=0.1)

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(pts[:, 0], pts[:, 1], s=0.5, alpha=0.05, color="0.6")
draw_ellipses(ax, show, evals, evecs, color="crimson", lw=1.2)
ax.set_aspect("equal")
ax.set_title("Learned metric at h = 0.1")
plt.show()

## Geometry across scales: the bandwidth $h$

The model is conditioned on $h$, so one network holds the geometry at every
scale. At small $h$ the ellipses follow the wiggles. Once $h$ passes the
wiggle wavelength the metric only sees the coarse circle, the tangents rotate
towards the circle's, and eventually the metric stops being rank-1 at all.

In [ ]:
H_VALUES = [0.03, 0.1, 0.6]

fig, axes = plt.subplots(1, len(H_VALUES), figsize=(4.4 * len(H_VALUES), 4.4))
for ax, h_val in zip(axes, H_VALUES):
    ev, evec = metric_eigs(show, h_value=h_val)
    ax.scatter(pts[:, 0], pts[:, 1], s=0.5, alpha=0.05, color="0.6")
    draw_ellipses(ax, show, ev, evec, color="crimson", lw=1.2)
    ax.set_aspect("equal")
    ax.set_title(f"h = {h_val:g}")
plt.tight_layout()
plt.show()

## Bonus: low-rank training with rank 1

The low-rank loss never builds the $D \times D$ metric, and Theorem 4.4 says
a rank of $r \ge 2d - 1$ is always enough. For a 1-dimensional manifold that
is $r = 1$. The tangent bundle of our curve is trivial (one smooth vector
field spans it everywhere), so a single output vector $u(x, h)$ with
$G = u u^\top$ can represent the exact metric. Rank 1 also changes what is
being learned: $G$ is rank-1 by construction, with $\lambda_2 = 0$
identically, so the only remaining question is whether $u$ points along the
tangent. Compared to the sphere footnote in the paper: $S^2$ is not
parallelizable, so there $r = 2$ would not be enough.

In [ ]:
cfg_r1 = {
    **cfg,
    "model": {"name": "mlp", "params": {**cfg["model"]["params"], "rank": 1}},
}
seed_everything(1, workers=True)
model_r1 = MMSystem(cfg_r1)

trainer_r1 = Trainer(
    max_epochs=3000,
    accelerator="gpu" if DEVICE == "cuda" else "cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)
trainer_r1.fit(
    model_r1,
    train_dataloaders=DataLoader(
        TensorDataset(pts, torch.zeros(len(pts))), batch_size=256, shuffle=True
    ),
)
model_r1 = model_r1.to(DEVICE).eval()


def rank1_field(points, h_value):
    """The single learned vector u(x, h), so that G = u u^T."""
    with torch.no_grad():
        u = model_r1(
            torch.full((len(points),), h_value, device=DEVICE), points.to(DEVICE)
        )
    return u.squeeze(1).cpu()  # [N, 2]

In [ ]:
u = rank1_field(show, h_value=0.1)
G = u.unsqueeze(-1) @ u.unsqueeze(-2)  # [N, 2, 2]
evals_r1, evecs_r1 = eigs_desc(G)
evals_r1 = evals_r1.clamp_min(0)  # Remove tiny negative roundoff errors.

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(pts[:, 0], pts[:, 1], s=0.5, alpha=0.05, color="0.6")
draw_ellipses(
    ax, show, evals_r1, evecs_r1,
    color="darkorange", lw=1.2,
)
ax.set_aspect("equal")
ax.set_title("Rank-1 model at h = 0.1")
plt.show()

## Where to go next

- The sphere experiment from the paper is the same recipe at a larger scale
  (a `d = 8` sphere in $\mathbb{R}^{64}$, rank-16 MLP):
  `python scripts/train.py --config-name sphere`.
- The image experiments swap the MLP for a UNet and, for MNIST, CelebA and
  FFHQ, use the mean-centred loss with a pretrained score model. The README
  describes the two-step recipe.
- For working with an already trained model (eigenvector visualization,
  spectra, intrinsic dimension), see `notebooks/inference_tutorial.ipynb`.